In [ ]:
from notebookutils import mssparkutils

result = mssparkutils.notebook.run(notebookName)
print("Result:", result)

In [2]:
container_Path=''

StatementMeta(, 7dd84a95-e1f8-48f1-87ff-f216832a772e, 4, Finished, Available, Finished)

In [ ]:
from pyspark.sql import functions as F

# --- CONFIG -------------------------------------------------------------
bronze_root_prefix = "Files/"   # e.g. "Files/"

# Read meta.
meta_df = spark.table("ProcessingSteps")

# Keep only active Silver rows
meta_silver = (
    meta_df
    .filter((F.col("Layer") == "Silver") & (F.col("IsCurrent") == 1) & (F.col("ContainerPath") == containerPath))
)

display(meta_silver)


In [4]:
def build_select_list(col_string: str) -> str:
    """
    Convert '[Col1], [Col2], [Col3]' → '`Col1`, `Col2`, `Col3`'
    for use in Spark SQL SELECT.
    """
    if col_string is None:
        raise ValueError("ColumnNames is NULL in metadata.")

    parts = col_string.split(",")
    cleaned_cols = []
    for p in parts:
        c = p.strip()
        # remove leading/trailing square brackets
        if c.startswith("[") and c.endswith("]"):
            c = c[1:-1]
        cleaned_cols.append(f"{c}")

    return ", ".join(cleaned_cols)


StatementMeta(, 7dd84a95-e1f8-48f1-87ff-f216832a772e, 6, Finished, Available, Finished)

In [ ]:
import re

def safe_view_name(name: str) -> str:
    # Replace anything not alphanumeric or underscore with underscore
    return re.sub(r"[^0-9a-zA-Z_]", "_", name)

# Fix for INT96 "ancient timestamp" parquet reads (choose LEGACY or CORRECTED)
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")

for row in meta_silver.collect():
    container_path = row["ContainerPath"]
    target_table   = row["TargetTableName"]        # e.g. 'SCHEMA.TABLE'
    col_names_raw  = row["ColumnNames"]
    partition_col  = row["PartitionedColumn"]

    full_path = f"{bronze_root_prefix}{container_path}/*.parquet"
    print(f"\n=== Processing {target_table} from {full_path} ===")

    # 1) Read bronze parquet (with the rebase config already set above)
    df_bronze = spark.read.parquet(full_path)

    # 2) Create a SAFE temp view name (no dots, spaces, etc.)
    tmp_view_name = f"tmp_{safe_view_name(target_table)}"
    df_bronze.createOrReplaceTempView(tmp_view_name)

    # 3) Build SELECT list from metadata
    select_list = build_select_list(col_names_raw)

    # 4) Run SQL using the safe temp view name
    sql_query = f"SELECT {select_list} FROM `{tmp_view_name}`"
    print(f"Running query:\n{sql_query}")

    df_silver = spark.sql(sql_query)

    # 5) Write to schema.table
    parts = target_table.split(".")
    if len(parts) != 2:
        raise ValueError(f"TargetTableName must be 'schema.table', got: {target_table}")

    schema, table = parts[0], parts[1]
    full_table_name = f"{schema}.{table}"  # DO NOT backtick-quote for saveAsTable

    writer = df_silver.write.mode("overwrite")
    if partition_col and partition_col.strip():
        writer = writer.partitionBy(partition_col)

    writer.saveAsTable(full_table_name)

    print(f"Saved silver table: {full_table_name}")
